# COOP SF Crime Data Exploration

This dataset provided by COOP contains 153,874 crime incidents reported in San Francisco during the year 2018

Each row contains valuable information such as time of day, day of the week, incident category, SF neighborhood with specific intersection, and more detailed location data

I'll be looking at and testing a few things to explore and find statistics about this data. 

One thing I want to look at is the rate of violent vs non-violent crimes in the Tenderloin/Mission areas (these areas have the highest amount of incidents from this list)

In [5]:
import pandas as pd
sf_crime_data = pd.read_csv('datasets/COOP/SF_CRIME_DATA_RAW.csv')

### Running some basic data information commands

In [6]:
sf_crime_data.shape

(153874, 26)

In [28]:
sf_crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153874 entries, 0 to 153873
Data columns (total 26 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Incident Datetime        153874 non-null  object 
 1   Incident Date            153874 non-null  object 
 2   Incident Time            153874 non-null  object 
 3   Incident Year            153874 non-null  int64  
 4   Incident Day of Week     153874 non-null  object 
 5   Report Datetime          153874 non-null  object 
 6   Row ID                   153874 non-null  int64  
 7   Incident ID              153874 non-null  int64  
 8   Incident Number          153874 non-null  int64  
 9   CAD Number               117827 non-null  float64
 10  Report Type Code         153874 non-null  object 
 11  Report Type Description  153874 non-null  object 
 12  Filed Online             33909 non-null   object 
 13  Incident Code            153874 non-null  int64  
 14  Inci

In [7]:
sf_crime_data.head()

,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Row ID,Incident ID,Incident Number,CAD Number,...,Incident Description,Resolution,Intersection,CNN,Police District,Analysis Neighborhood,Supervisor District,Latitude,Longitude,point
0,2018/01/01 9:26:00 AM,2018/01/01,9:26,2018,Monday,2018/01/01 9:27:00 AM,61893007041,618930,171052174,173641140.0,...,"Vehicle, Recovered, Auto",Open or Active,03RD ST \ HOLLISTER AVE,20471000.0,Southern,Bayview Hunters Point,10.0,37.721716,-122.395944,"(37.72171587946975, -122.39594382884452)"
1,2018/01/01 2:30:00 AM,2018/01/01,2:30,2018,Monday,2018/01/01 8:21:00 AM,61893105041,618931,180000768,180010668.0,...,"Burglary, Residence, Forcible Entry",Open or Active,LISBON ST \ PERSIA AVE,21719000.0,Ingleside,Excelsior,11.0,37.722000,-122.433606,"(37.722000219874225, -122.43360633930074)"
2,2018/01/01 10:00:00 AM,2018/01/01,10:00,2018,Monday,2018/01/01 10:20:00 AM,61893275000,618932,180000605,180010893.0,...,Found Person,Open or Active,VAN NESS AVE \ WILLOW ST,25189000.0,Northern,Tenderloin,6.0,37.783370,-122.420832,"(37.78337048750076, -122.42083185184009)"
3,2018/01/01 10:03:00 AM,2018/01/01,10:03,2018,Monday,2018/01/01 10:04:00 AM,61893565015,618935,180000887,180011579.0,...,"Driving, No License Issued",Cite or Arrest Adult,BRAZIL AVE \ MISSION ST,21769000.0,Ingleside,Outer Mission,11.0,37.724683,-122.434798,"(37.72468255342173, -122.43479841474401)"
4,2018/01/01 9:01:00 AM,2018/01/01,9:01,2018,Monday,2018/01/01 9:39:00 AM,61893607041,618936,171052958,180011403.0,...,"Vehicle, Recovered, Auto",Open or Active,CUSTOM HOUSE PL \ JACKSON ST,24709000.0,Central,Chinatown,3.0,37.796698,-122.401294,"(37.796698028315056, -122.40129440446798)"


### Grouping incidents by Neighborhood and by day of the week 

We see here that the most incidents were reported in the Mission and the Tenderloin. There is a slight drop off from number 4 to 5 on this list (SOMA to Bayview Hunters Point)

We also see that the most incidents were reported on Fridays, followed by Wednesdays

In [84]:
sf_crime_data.value_counts(subset=['Analysis Neighborhood'])

Analysis Neighborhood         
Mission                           17663
Tenderloin                        15101
Financial District/South Beach    13977
South of Market                   13148
Bayview Hunters Point              8192
North Beach                        5079
Western Addition                   4541
Castro/Upper Market                4519
Sunset/Parkside                    4436
Nob Hill                           3926
Hayes Valley                       3667
Russian Hill                       3511
Marina                             3102
Chinatown                          2934
Outer Richmond                     2929
West of Twin Peaks                 2795
Mission Bay                        2686
Haight Ashbury                     2498
Bernal Heights                     2472
Excelsior                          2409
Potrero Hill                       2378
Pacific Heights                    2270
Outer Mission                      1980
Inner Richmond                     1909
Lone Moun

In [23]:
sf_crime_data.value_counts('Incident Day of Week')

Incident Day of Week
Friday       23442
Wednesday    22544
Saturday     22240
Monday       21868
Thursday     21806
Tuesday      21745
Sunday       20229
Name: count, dtype: int64

### Now trying some time specific analysis

Wanted to see what areas were being hit with the most incidents for each day. 

In [24]:
most_freq_by_day = sf_crime_data.groupby('Incident Day of Week')['Analysis Neighborhood'].agg(lambda x: x.value_counts().index[0])
print(most_freq_by_day)

Incident Day of Week
Friday       Mission
Monday       Mission
Saturday     Mission
Sunday       Mission
Thursday     Mission
Tuesday      Mission
Wednesday    Mission
Name: Analysis Neighborhood, dtype: str


Now I want to see the numbers on each day for Mission specifically

In [33]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

mission_data = sf_crime_data[sf_crime_data['Analysis Neighborhood'] == 'Mission']
mission_freq_by_day = mission_data['Incident Day of Week'].value_counts()

mission_freq_by_day = mission_freq_by_day.reindex(days)
print(mission_freq_by_day)


Incident Day of Week
Monday       2488
Tuesday      2491
Wednesday    2667
Thursday     2425
Friday       2571
Saturday     2531
Sunday       2490
Name: count, dtype: int64


This didn't help me so I wanted to sort it by time and see what's going on

In [27]:
#getting each time categorized into each hour of the day
pd.to_datetime(sf_crime_data['Incident Time'])
sf_crime_data['hour'] = pd.to_datetime(sf_crime_data['Incident Time']).dt.hour

C:\Users\Fires\AppData\Local\Temp\ipykernel_18272\4251203464.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sf_crime_data['Incident Time'])
C:\Users\Fires\AppData\Local\Temp\ipykernel_18272\4251203464.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sf_crime_data['hour'] = pd.to_datetime(sf_crime_data['Incident Time']).dt.hour


In [28]:
most_freq_crime_by_hour = sf_crime_data.groupby('hour')['Analysis Neighborhood'].agg(lambda x: x.value_counts().index[0])
print(most_freq_crime_by_hour)

hour
0                            Mission
1                            Mission
2                            Mission
3                            Mission
4                            Mission
5                            Mission
6                         Tenderloin
7                         Tenderloin
8                         Tenderloin
9                         Tenderloin
10                   South of Market
11                        Tenderloin
12                           Mission
13                        Tenderloin
14                           Mission
15    Financial District/South Beach
16                        Tenderloin
17    Financial District/South Beach
18                           Mission
19                           Mission
20                           Mission
21                           Mission
22                           Mission
23                           Mission
Name: Analysis Neighborhood, dtype: str


From the last cell we see that a lot of overnight incidents happen in the Mission. Since the Mission has the highest report rate, I wanted to see what the daytime incident rate looked like for all neighborhoods

In [29]:
#making var for daytime incidents
daytime_crime_data = sf_crime_data[(sf_crime_data['hour'] > 8) & (sf_crime_data['hour'] < 17)]

In [30]:
#looking at the hourly data but just from 9-5, I could change the time range too

#This data might be useful for looking at bakery location/opening hours

daytime_most_freq_crime_by_hour = daytime_crime_data.groupby('hour')['Analysis Neighborhood'].agg(lambda x: x.value_counts().index[0])
print(daytime_most_freq_crime_by_hour)

hour
9                         Tenderloin
10                   South of Market
11                        Tenderloin
12                           Mission
13                        Tenderloin
14                           Mission
15    Financial District/South Beach
16                        Tenderloin
Name: Analysis Neighborhood, dtype: str


In [90]:
#New day of the week data has more variety in information without the overnight incidents
daytime_most_freq_by_day = daytime_crime_data.groupby('Incident Day of Week')['Analysis Neighborhood'].agg(lambda x: x.value_counts().index[0])
print(daytime_most_freq_by_day)

Incident Day of Week
Friday                           Tenderloin
Monday                              Mission
Saturday     Financial District/South Beach
Sunday                           Tenderloin
Thursday                         Tenderloin
Tuesday                             Mission
Wednesday                        Tenderloin
Name: Analysis Neighborhood, dtype: object


In [31]:
#revisiting daytime incident numbers
daytime_crime_data.value_counts(subset=['Analysis Neighborhood','Incident Category'])

Analysis Neighborhood           Incident Category
Financial District/South Beach  Larceny Theft        2706
Mission                         Larceny Theft        1465
North Beach                     Larceny Theft        1252
Tenderloin                      Drug Offense         1178
                                Larceny Theft        1124
                                                     ... 
Portola                         Fire Report             1
Inner Richmond                  Weapons Offense         1
Outer Mission                   Prostitution            1
Treasure Island                 Weapons Offense         1
Haight Ashbury                  Rape                    1
Name: count, Length: 1352, dtype: int64

Logging off for now, but next I want to look at some specific Qs that we came up with